In [1]:
# ==============================================================
# Indian Pines 高光谱消融实验：精细分类 (16类) vs 粗粒度分类 (Crop)
# 综合对比测试：7 种 SOTA MOEAs (包含 MOFS-REPLS 和 DREA-FS 专版)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy scipy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import random
import scipy.io as sio

# --- Sklearn ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排序，保证表格列顺序一致
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration & Data Utils
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def _load_mat_first_var(path):
    mat = sio.loadmat(path)
    for k in mat.keys():
        if not k.startswith('__'): return mat[k]
    raise ValueError(f"No valid variable found in {path}")

def make_crop_label_from_indian_pines(y_fine):
    crop_classes = [1, 2, 3, 4, 9, 10, 11, 12, 14]
    return np.isin(y_fine, crop_classes).astype(int)

def stratified_subsample_per_class(X, y, n_per_class=20, seed=42, keep_all_if_less=True):
    np.random.seed(seed)
    idx_selected = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        if len(c_idx) >= n_per_class:
            selected = np.random.choice(c_idx, n_per_class, replace=False)
        else:
            selected = c_idx if keep_all_if_less else np.random.choice(c_idx, n_per_class, replace=True)
        idx_selected.extend(selected)
    
    idx_selected = np.array(idx_selected)
    np.random.shuffle(idx_selected)
    return X[idx_selected], y[idx_selected], idx_selected

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用随机森林特征重要性)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Task1_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task1'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Max_Task2_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task2'][sel])) + _size_penalty(len(sel), len(x))

class HierarchicalFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Task1_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Max_Task2_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 2. 定制算子 (MOFS-REPLS & DREA-FS)
# ==============================================================
class REPLSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        
        weights = problem.opts['imp_task1']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0] 
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                
                p_nearest_indices = np.argsort(dists)[:self.P]
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0:
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66:
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False

            infills.set("X", X_inf)
        return infills

class DREAFSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_task1 = problem.opts['imp_task1']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_task1[imp_task1 > 0]) if np.any(imp_task1 > 0) else 0
        important_features = np.where(imp_task1 >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask 
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
                    
        return X_init

class DREA_FS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 3. Core Evaluation Routine
# ==============================================================
def calc_mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

def run_ip_unified_benchmark(features, task1_labels, task2_labels, n_runs=10):
    all_results = []
    
    print("="*70)
    print(f"Starting INDIAN PINES UNIFIED MOEAs BENCHMARK: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Run {run_idx}/{n_runs}...")
        
        le_t1 = LabelEncoder()
        task1_labels_encoded = le_t1.fit_transform(task1_labels)
        
        xtrain, xtest, ytrain, ytest, y2_tr, y2_te = train_test_split(
            features, task1_labels_encoded, task2_labels, 
            test_size=0.3, stratify=task1_labels_encoded, random_state=current_seed
        )
        
        print("  Pre-computing Feature Importances (Random Forest)...")
        rf_task1 = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf_task1.fit(xtrain, ytrain)
        imp_task1 = rf_task1.feature_importances_
        
        rf_task2 = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf_task2.fit(xtrain, y2_tr)
        imp_task2 = rf_task2.feature_importances_
        
        opts = {'xt_in': xtrain, 'imp_task1': imp_task1, 'imp_task2': imp_task2}
        problem = HierarchicalFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs)
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
            "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
        }

        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else np.array([0])

        clf = LinearDiscriminantAnalysis()

        print("  [Optimizing] Running MOEAs...")
        for algo_name, algo in algorithms.items():
            
            # --- 记录运算时间 ---
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            # --------------------
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)

            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                if len(idx) == 0: continue
                
                X_tr_sel, X_te_sel = xtrain[:, idx], xtest[:, idx]

                # Task 1 Evaluation (Fine-grained)
                clf.fit(X_tr_sel, ytrain)
                acc_t1 = clf.score(X_te_sel, ytest)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task1_Fine", "Value": acc_t1,
                    "Time_Seconds": elapsed_time
                })

                # Task 2 Evaluation (Crop vs Non-Crop)
                clf.fit(X_tr_sel, y2_tr)
                acc_t2 = clf.score(X_te_sel, y2_te)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task2_Crop", "Value": acc_t2,
                    "Time_Seconds": elapsed_time
                })

    # ==============================================================
    # 4. Data Summary & Output
    # ==============================================================
    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("                 UNIFIED MOEAs TABLES [Mean ± Std]")
    print("="*80)
    
    # Task 1 准确率表格
    df_t1 = df_results[df_results["Task"] == "Predict_Task1_Fine"]
    table_t1 = df_t1.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t1 = table_t1[[m for m in METHOD_ORDER if m in table_t1.columns]]
    print("\n[TABLE 1] Task 1: Fine-grained (16-Class) Accuracy")
    print(table_t1)
    table_t1.to_csv("Unified_IP_Task1_LDA.csv")
    
    # Task 2 准确率表格
    df_t2 = df_results[df_results["Task"] == "Predict_Task2_Crop"]
    table_t2 = df_t2.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t2 = table_t2[[m for m in METHOD_ORDER if m in table_t2.columns]]
    print("\n[TABLE 2] Task 2: Coarse (Crop/Non-Crop) Accuracy")
    print(table_t2)
    table_t2.to_csv("Unified_IP_Task2_LDA.csv")
    
    # 运行时间表格
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Unified_IP_Execution_Time.csv")
    
    print("\n[+] 印度松林数据集 7大算法统一基准运行完毕！表格已生成。")

# ==============================================================
# 5. Main Execution
# ==============================================================
if __name__ == "__main__":
    DATA_CUBE_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_corrected.mat"
    GT_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_gt.mat"
    SUBSAMPLE_PER_CLASS = 20   
    KEEP_ALL_IF_LESS = True

    try:
        if not os.path.exists(DATA_CUBE_PATH) or not os.path.exists(GT_PATH):
            raise FileNotFoundError("Indian Pines .mat files not found in datasets/ip/ directory.")
            
        print("Loading Indian Pines (.mat)...")
        img = _load_mat_first_var(DATA_CUBE_PATH)
        lab = _load_mat_first_var(GT_PATH)

        if lab.ndim == 3 and lab.shape[-1] == 1: lab = lab[:, :, 0]
        lab = lab.astype(int)

        mask = lab > 0
        X = img[mask].astype(float)
        y_fine = lab[mask].astype(int)
        y_crop = make_crop_label_from_indian_pines(y_fine)

        if SUBSAMPLE_PER_CLASS is not None:
            X, y_fine, idx = stratified_subsample_per_class(
                X, y_fine, n_per_class=SUBSAMPLE_PER_CLASS, seed=42, keep_all_if_less=KEEP_ALL_IF_LESS
            )
            y_crop = y_crop[idx]

        X = normalize(X + 1e-8, norm="l1", axis=1)

        run_ip_unified_benchmark(X, y_fine, y_crop, n_runs=30) 

    except Exception as e:
        print(f"\nError encountered: {e}")
        print("Running with Safe Mock Data for architecture verification...")
        set_seed(42)
        X_mock = np.random.rand(500, 200)
        y_fine_mock = np.random.randint(1, 17, 500)
        y_crop_mock = make_crop_label_from_indian_pines(y_fine_mock)
        X_mock = normalize(X_mock + 1e-8, norm="l1", axis=1)
        
        run_ip_unified_benchmark(X_mock, y_fine_mock, y_crop_mock, n_runs=2)

Loading Indian Pines (.mat)...
Starting INDIAN PINES UNIFIED MOEAs BENCHMARK: 30 Runs

>>> Run 1/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 2/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 3/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 4/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 5/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 6/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 7/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 8/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 9/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimiz

In [2]:
# ==============================================================
# Indian Pines 高光谱消融实验：精细分类 (16类) vs 粗粒度分类 (Crop)
# 综合对比测试：7 种 SOTA MOEAs (包含 MOFS-REPLS 和 DREA-FS 专版)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy scipy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import random
import scipy.io as sio

# --- Sklearn ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排序，保证表格列顺序一致
METHOD_ORDER = [ "MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration & Data Utils
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def _load_mat_first_var(path):
    mat = sio.loadmat(path)
    for k in mat.keys():
        if not k.startswith('__'): return mat[k]
    raise ValueError(f"No valid variable found in {path}")

def make_crop_label_from_indian_pines(y_fine):
    crop_classes = [1, 2, 3, 4, 9, 10, 11, 12, 14]
    return np.isin(y_fine, crop_classes).astype(int)

def stratified_subsample_per_class(X, y, n_per_class=20, seed=42, keep_all_if_less=True):
    np.random.seed(seed)
    idx_selected = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        if len(c_idx) >= n_per_class:
            selected = np.random.choice(c_idx, n_per_class, replace=False)
        else:
            selected = c_idx if keep_all_if_less else np.random.choice(c_idx, n_per_class, replace=True)
        idx_selected.extend(selected)
    
    idx_selected = np.array(idx_selected)
    np.random.shuffle(idx_selected)
    return X[idx_selected], y[idx_selected], idx_selected

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用随机森林特征重要性)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Task1_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task1'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Max_Task2_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task2'][sel])) + _size_penalty(len(sel), len(x))

class HierarchicalFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Task1_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Max_Task2_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 2. 定制算子 (MOFS-REPLS & DREA-FS)
# ==============================================================
class REPLSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        
        weights = problem.opts['imp_task1']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0] 
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                
                p_nearest_indices = np.argsort(dists)[:self.P]
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0:
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66:
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False

            infills.set("X", X_inf)
        return infills

class DREAFSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_task1 = problem.opts['imp_task1']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_task1[imp_task1 > 0]) if np.any(imp_task1 > 0) else 0
        important_features = np.where(imp_task1 >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask 
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
                    
        return X_init

class DREA_FS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 3. Core Evaluation Routine
# ==============================================================
def calc_mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

def run_ip_unified_benchmark(features, task1_labels, task2_labels, n_runs=10):
    all_results = []
    
    print("="*70)
    print(f"Starting INDIAN PINES UNIFIED MOEAs BENCHMARK: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Run {run_idx}/{n_runs}...")
        
        le_t1 = LabelEncoder()
        task1_labels_encoded = le_t1.fit_transform(task1_labels)
        
        xtrain, xtest, ytrain, ytest, y2_tr, y2_te = train_test_split(
            features, task1_labels_encoded, task2_labels, 
            test_size=0.3, stratify=task1_labels_encoded, random_state=current_seed
        )
        
        print("  Pre-computing Feature Importances (Random Forest)...")
        rf_task1 = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf_task1.fit(xtrain, ytrain)
        imp_task1 = rf_task1.feature_importances_
        
        rf_task2 = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
        rf_task2.fit(xtrain, y2_tr)
        imp_task2 = rf_task2.feature_importances_
        
        opts = {'xt_in': xtrain, 'imp_task1': imp_task1, 'imp_task2': imp_task2}
        problem = HierarchicalFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs)
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
            "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
        }

        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else np.array([0])

        clf = LinearDiscriminantAnalysis()

        print("  [Optimizing] Running MOEAs...")
        for algo_name, algo in algorithms.items():
            
            # --- 记录运算时间 ---
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            # --------------------
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)

            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                if len(idx) == 0: continue
                
                X_tr_sel, X_te_sel = xtrain[:, idx], xtest[:, idx]

                # Task 1 Evaluation (Fine-grained)
                clf.fit(X_tr_sel, ytrain)
                acc_t1 = clf.score(X_te_sel, ytest)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task1_Fine", "Value": acc_t1,
                    "Time_Seconds": elapsed_time
                })

                # Task 2 Evaluation (Crop vs Non-Crop)
                clf.fit(X_tr_sel, y2_tr)
                acc_t2 = clf.score(X_te_sel, y2_te)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task2_Crop", "Value": acc_t2,
                    "Time_Seconds": elapsed_time
                })

    # ==============================================================
    # 4. Data Summary & Output
    # ==============================================================
    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("                 UNIFIED MOEAs TABLES [Mean ± Std]")
    print("="*80)
    
    # Task 1 准确率表格
    df_t1 = df_results[df_results["Task"] == "Predict_Task1_Fine"]
    table_t1 = df_t1.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t1 = table_t1[[m for m in METHOD_ORDER if m in table_t1.columns]]
    print("\n[TABLE 1] Task 1: Fine-grained (16-Class) Accuracy")
    print(table_t1)
    table_t1.to_csv("Unified_IP_Task1_LDA.csv")
    
    # Task 2 准确率表格
    df_t2 = df_results[df_results["Task"] == "Predict_Task2_Crop"]
    table_t2 = df_t2.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t2 = table_t2[[m for m in METHOD_ORDER if m in table_t2.columns]]
    print("\n[TABLE 2] Task 2: Coarse (Crop/Non-Crop) Accuracy")
    print(table_t2)
    table_t2.to_csv("Unified_IP_Task2_LDA.csv")
    
    # 运行时间表格
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Unified_IP_Execution_Time.csv")
    
    print("\n[+] 印度松林数据集 7大算法统一基准运行完毕！表格已生成。")

# ==============================================================
# 5. Main Execution
# ==============================================================
if __name__ == "__main__":
    DATA_CUBE_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_corrected.mat"
    GT_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_gt.mat"
    SUBSAMPLE_PER_CLASS = 20   
    KEEP_ALL_IF_LESS = True

    try:
        if not os.path.exists(DATA_CUBE_PATH) or not os.path.exists(GT_PATH):
            raise FileNotFoundError("Indian Pines .mat files not found in datasets/ip/ directory.")
            
        print("Loading Indian Pines (.mat)...")
        img = _load_mat_first_var(DATA_CUBE_PATH)
        lab = _load_mat_first_var(GT_PATH)

        if lab.ndim == 3 and lab.shape[-1] == 1: lab = lab[:, :, 0]
        lab = lab.astype(int)

        mask = lab > 0
        X = img[mask].astype(float)
        y_fine = lab[mask].astype(int)
        y_crop = make_crop_label_from_indian_pines(y_fine)

        if SUBSAMPLE_PER_CLASS is not None:
            X, y_fine, idx = stratified_subsample_per_class(
                X, y_fine, n_per_class=SUBSAMPLE_PER_CLASS, seed=42, keep_all_if_less=KEEP_ALL_IF_LESS
            )
            y_crop = y_crop[idx]

        X = normalize(X + 1e-8, norm="l1", axis=1)

        run_ip_unified_benchmark(X, y_fine, y_crop, n_runs=30) 

    except Exception as e:
        print(f"\nError encountered: {e}")
        print("Running with Safe Mock Data for architecture verification...")
        set_seed(42)
        X_mock = np.random.rand(500, 200)
        y_fine_mock = np.random.randint(1, 17, 500)
        y_crop_mock = make_crop_label_from_indian_pines(y_fine_mock)
        X_mock = normalize(X_mock + 1e-8, norm="l1", axis=1)
        
        run_ip_unified_benchmark(X_mock, y_fine_mock, y_crop_mock, n_runs=2)

Loading Indian Pines (.mat)...
Starting INDIAN PINES UNIFIED MOEAs BENCHMARK: 30 Runs

>>> Run 1/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 2/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 3/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 4/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 5/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 6/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 7/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 8/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimizing] Running MOEAs...

>>> Run 9/30...
  Pre-computing Feature Importances (Random Forest)...
  [Optimiz

In [4]:
# ==============================================================
# 纯净版 Baseline 实验：Indian Pines (Hierarchical: Fine vs Coarse/Crop)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy scipy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import random
import scipy.io as sio

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排序，保证表格列顺序一致
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration & Data Utils
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def _load_mat_first_var(path):
    mat = sio.loadmat(path)
    for k in mat.keys():
        if not k.startswith('__'): return mat[k]
    raise ValueError(f"No valid variable found in {path}")

def make_crop_label_from_indian_pines(y_fine):
    crop_classes = [1, 2, 3, 4, 9, 10, 11, 12, 14]
    return np.isin(y_fine, crop_classes).astype(int)

def stratified_subsample_per_class(X, y, n_per_class=20, seed=42, keep_all_if_less=True):
    np.random.seed(seed)
    idx_selected = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        if len(c_idx) >= n_per_class:
            selected = np.random.choice(c_idx, n_per_class, replace=False)
        else:
            selected = c_idx if keep_all_if_less else np.random.choice(c_idx, n_per_class, replace=True)
        idx_selected.extend(selected)
    
    idx_selected = np.array(idx_selected)
    np.random.shuffle(idx_selected)
    return X[idx_selected], y[idx_selected], idx_selected

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息评估特征)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Task1_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task1'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Max_Task2_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task2'][sel])) + _size_penalty(len(sel), len(x))

class HierarchicalFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：最大化 Task1 互信息，最小化冗余，最大化 Task2 互信息
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Task1_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Max_Task2_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 2. Core Evaluation Routine
# ==============================================================
def calc_mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

def run_ip_unified_benchmark(features, task1_labels, task2_labels, n_runs=10):
    all_results = []
    
    print("="*70)
    print(f"Starting INDIAN PINES PURE MOEAs BENCHMARK: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Run {run_idx}/{n_runs}...")
        
        le_t1 = LabelEncoder()
        task1_labels_encoded = le_t1.fit_transform(task1_labels)
        
        xtrain, xtest, ytrain, ytest, y2_tr, y2_te = train_test_split(
            features, task1_labels_encoded, task2_labels, 
            test_size=0.3, stratify=task1_labels_encoded, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # --- 核心修正：统一使用 mutual_info_classif 保持公平对比 ---
        imp_task1 = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
        imp_task2 = mutual_info_classif(xtrain, y2_tr, random_state=current_seed)
        
        opts = {'xt_in': xtrain, 'imp_task1': imp_task1, 'imp_task2': imp_task2}
        problem = HierarchicalFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
        }

        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else np.array([0])

        clf = LinearDiscriminantAnalysis()

        print("  [Optimizing] Running Pure MOEAs...")
        for algo_name, algo in algorithms.items():
            
            # --- 记录运算时间 ---
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            # --------------------
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)

            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                if len(idx) == 0: continue
                
                X_tr_sel, X_te_sel = xtrain[:, idx], xtest[:, idx]

                # Task 1 Evaluation (Fine-grained)
                clf.fit(X_tr_sel, ytrain)
                acc_t1 = clf.score(X_te_sel, ytest)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task1_Fine", "Value": acc_t1,
                    "Time_Seconds": elapsed_time
                })

                # Task 2 Evaluation (Crop vs Non-Crop)
                clf.fit(X_tr_sel, y2_tr)
                acc_t2 = clf.score(X_te_sel, y2_te)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task2_Crop", "Value": acc_t2,
                    "Time_Seconds": elapsed_time
                })

    # ==============================================================
    # 3. Data Summary & Output
    # ==============================================================
    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("                 PURE MOEAs TABLES [Mean ± Std]")
    print("="*80)
    
    # Task 1 准确率表格
    df_t1 = df_results[df_results["Task"] == "Predict_Task1_Fine"]
    table_t1 = df_t1.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t1 = table_t1[[m for m in METHOD_ORDER if m in table_t1.columns]]
    print("\n[TABLE 1] Task 1: Fine-grained (16-Class) Accuracy")
    print(table_t1)
    table_t1.to_csv("Pure_MOEA_IP_Task1_LDA.csv")
    
    # Task 2 准确率表格
    df_t2 = df_results[df_results["Task"] == "Predict_Task2_Crop"]
    table_t2 = df_t2.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t2 = table_t2[[m for m in METHOD_ORDER if m in table_t2.columns]]
    print("\n[TABLE 2] Task 2: Coarse (Crop/Non-Crop) Accuracy")
    print(table_t2)
    table_t2.to_csv("Pure_MOEA_IP_Task2_LDA.csv")
    
    # 运行时间表格
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Pure_MOEA_IP_Execution_Time.csv")
    
    print("\n[+] 印度松林数据集 7大纯血算法统一基准运行完毕！表格已生成。")

# ==============================================================
# 4. Main Execution
# ==============================================================
if __name__ == "__main__":
    DATA_CUBE_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_corrected.mat"
    GT_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_gt.mat"
    SUBSAMPLE_PER_CLASS = 20   
    KEEP_ALL_IF_LESS = True

    try:
        if not os.path.exists(DATA_CUBE_PATH) or not os.path.exists(GT_PATH):
            raise FileNotFoundError("Indian Pines .mat files not found. Please check paths.")
            
        print("Loading Indian Pines (.mat)...")
        img = _load_mat_first_var(DATA_CUBE_PATH)
        lab = _load_mat_first_var(GT_PATH)

        if lab.ndim == 3 and lab.shape[-1] == 1: lab = lab[:, :, 0]
        lab = lab.astype(int)

        mask = lab > 0
        X = img[mask].astype(float)
        y_fine = lab[mask].astype(int)
        y_crop = make_crop_label_from_indian_pines(y_fine)

        if SUBSAMPLE_PER_CLASS is not None:
            X, y_fine, idx = stratified_subsample_per_class(
                X, y_fine, n_per_class=SUBSAMPLE_PER_CLASS, seed=42, keep_all_if_less=KEEP_ALL_IF_LESS
            )
            y_crop = y_crop[idx]

        X = normalize(X + 1e-8, norm="l1", axis=1)

        run_ip_unified_benchmark(X, y_fine, y_crop, n_runs=30) 

    except Exception as e:
        print(f"\nError encountered: {e}")
        print("Running with Safe Mock Data for architecture verification...")
        set_seed(42)
        X_mock = np.random.rand(500, 200)
        y_fine_mock = np.random.randint(1, 17, 500)
        y_crop_mock = make_crop_label_from_indian_pines(y_fine_mock)
        X_mock = normalize(X_mock + 1e-8, norm="l1", axis=1)
        
        run_ip_unified_benchmark(X_mock, y_fine_mock, y_crop_mock, n_runs=2)

Loading Indian Pines (.mat)...
Starting INDIAN PINES PURE MOEAs BENCHMARK: 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Run

In [6]:
# ==============================================================
# 纯净版 Baseline 实验：Indian Pines (Hierarchical: Fine vs Coarse/Crop)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy scipy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import random
import scipy.io as sio

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排序，保证表格列顺序一致
METHOD_ORDER = ["AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration & Data Utils
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def _load_mat_first_var(path):
    mat = sio.loadmat(path)
    for k in mat.keys():
        if not k.startswith('__'): return mat[k]
    raise ValueError(f"No valid variable found in {path}")

def make_crop_label_from_indian_pines(y_fine):
    crop_classes = [1, 2, 3, 4, 9, 10, 11, 12, 14]
    return np.isin(y_fine, crop_classes).astype(int)

def stratified_subsample_per_class(X, y, n_per_class=20, seed=42, keep_all_if_less=True):
    np.random.seed(seed)
    idx_selected = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        if len(c_idx) >= n_per_class:
            selected = np.random.choice(c_idx, n_per_class, replace=False)
        else:
            selected = c_idx if keep_all_if_less else np.random.choice(c_idx, n_per_class, replace=True)
        idx_selected.extend(selected)
    
    idx_selected = np.array(idx_selected)
    np.random.shuffle(idx_selected)
    return X[idx_selected], y[idx_selected], idx_selected

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息评估特征)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Task1_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task1'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Max_Task2_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task2'][sel])) + _size_penalty(len(sel), len(x))

class HierarchicalFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：最大化 Task1 互信息，最小化冗余，最大化 Task2 互信息
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Task1_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Max_Task2_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 2. Core Evaluation Routine
# ==============================================================
def calc_mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

def run_ip_unified_benchmark(features, task1_labels, task2_labels, n_runs=10):
    all_results = []
    
    print("="*70)
    print(f"Starting INDIAN PINES PURE MOEAs BENCHMARK: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Run {run_idx}/{n_runs}...")
        
        le_t1 = LabelEncoder()
        task1_labels_encoded = le_t1.fit_transform(task1_labels)
        
        xtrain, xtest, ytrain, ytest, y2_tr, y2_te = train_test_split(
            features, task1_labels_encoded, task2_labels, 
            test_size=0.3, stratify=task1_labels_encoded, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # --- 核心修正：统一使用 mutual_info_classif 保持公平对比 ---
        imp_task1 = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
        imp_task2 = mutual_info_classif(xtrain, y2_tr, random_state=current_seed)
        
        opts = {'xt_in': xtrain, 'imp_task1': imp_task1, 'imp_task2': imp_task2}
        problem = HierarchicalFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        std_sampling = BinaryRandomSampling()
        
        algorithms = {
            "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
            "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
        }

        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else np.array([0])

        clf = LinearDiscriminantAnalysis()

        print("  [Optimizing] Running Pure MOEAs...")
        for algo_name, algo in algorithms.items():
            
            # --- 记录运算时间 ---
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            # --------------------
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)

            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                if len(idx) == 0: continue
                
                X_tr_sel, X_te_sel = xtrain[:, idx], xtest[:, idx]

                # Task 1 Evaluation (Fine-grained)
                clf.fit(X_tr_sel, ytrain)
                acc_t1 = clf.score(X_te_sel, ytest)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task1_Fine", "Value": acc_t1,
                    "Time_Seconds": elapsed_time
                })

                # Task 2 Evaluation (Crop vs Non-Crop)
                clf.fit(X_tr_sel, y2_tr)
                acc_t2 = clf.score(X_te_sel, y2_te)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task2_Crop", "Value": acc_t2,
                    "Time_Seconds": elapsed_time
                })

    # ==============================================================
    # 3. Data Summary & Output
    # ==============================================================
    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("                 PURE MOEAs TABLES [Mean ± Std]")
    print("="*80)
    
    # Task 1 准确率表格
    df_t1 = df_results[df_results["Task"] == "Predict_Task1_Fine"]
    table_t1 = df_t1.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t1 = table_t1[[m for m in METHOD_ORDER if m in table_t1.columns]]
    print("\n[TABLE 1] Task 1: Fine-grained (16-Class) Accuracy")
    print(table_t1)
    table_t1.to_csv("Pure_MOEA_IP_Task1_LDA.csv")
    
    # Task 2 准确率表格
    df_t2 = df_results[df_results["Task"] == "Predict_Task2_Crop"]
    table_t2 = df_t2.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t2 = table_t2[[m for m in METHOD_ORDER if m in table_t2.columns]]
    print("\n[TABLE 2] Task 2: Coarse (Crop/Non-Crop) Accuracy")
    print(table_t2)
    table_t2.to_csv("Pure_MOEA_IP_Task2_LDA.csv")
    
    # 运行时间表格
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Pure_MOEA_IP_Execution_Time.csv")
    
    print("\n[+] 印度松林数据集 7大纯血算法统一基准运行完毕！表格已生成。")

# ==============================================================
# 4. Main Execution
# ==============================================================
if __name__ == "__main__":
    DATA_CUBE_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_corrected.mat"
    GT_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_gt.mat"
    SUBSAMPLE_PER_CLASS = 20   
    KEEP_ALL_IF_LESS = True

    try:
        if not os.path.exists(DATA_CUBE_PATH) or not os.path.exists(GT_PATH):
            raise FileNotFoundError("Indian Pines .mat files not found. Please check paths.")
            
        print("Loading Indian Pines (.mat)...")
        img = _load_mat_first_var(DATA_CUBE_PATH)
        lab = _load_mat_first_var(GT_PATH)

        if lab.ndim == 3 and lab.shape[-1] == 1: lab = lab[:, :, 0]
        lab = lab.astype(int)

        mask = lab > 0
        X = img[mask].astype(float)
        y_fine = lab[mask].astype(int)
        y_crop = make_crop_label_from_indian_pines(y_fine)

        if SUBSAMPLE_PER_CLASS is not None:
            X, y_fine, idx = stratified_subsample_per_class(
                X, y_fine, n_per_class=SUBSAMPLE_PER_CLASS, seed=42, keep_all_if_less=KEEP_ALL_IF_LESS
            )
            y_crop = y_crop[idx]

        X = normalize(X + 1e-8, norm="l1", axis=1)

        run_ip_unified_benchmark(X, y_fine, y_crop, n_runs=30) 

    except Exception as e:
        print(f"\nError encountered: {e}")
        print("Running with Safe Mock Data for architecture verification...")
        set_seed(42)
        X_mock = np.random.rand(500, 200)
        y_fine_mock = np.random.randint(1, 17, 500)
        y_crop_mock = make_crop_label_from_indian_pines(y_fine_mock)
        X_mock = normalize(X_mock + 1e-8, norm="l1", axis=1)
        
        run_ip_unified_benchmark(X_mock, y_fine_mock, y_crop_mock, n_runs=2)

Loading Indian Pines (.mat)...
Starting INDIAN PINES PURE MOEAs BENCHMARK: 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Pure MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Run

In [5]:
# ==============================================================
# 定制版 Baseline 实验：Indian Pines (Hierarchical: Fine vs Coarse/Crop)
# 专项测试：仅运行 MOFS-REPLS 和 DREA-FS (包含专版采样器与算子)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy scipy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import random
import scipy.io as sio

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排序（仅保留定制算法）
METHOD_ORDER = ["MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration & Data Utils
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

def _load_mat_first_var(path):
    mat = sio.loadmat(path)
    for k in mat.keys():
        if not k.startswith('__'): return mat[k]
    raise ValueError(f"No valid variable found in {path}")

def make_crop_label_from_indian_pines(y_fine):
    crop_classes = [1, 2, 3, 4, 9, 10, 11, 12, 14]
    return np.isin(y_fine, crop_classes).astype(int)

def stratified_subsample_per_class(X, y, n_per_class=20, seed=42, keep_all_if_less=True):
    np.random.seed(seed)
    idx_selected = []
    classes = np.unique(y)
    for c in classes:
        c_idx = np.where(y == c)[0]
        if len(c_idx) >= n_per_class:
            selected = np.random.choice(c_idx, n_per_class, replace=False)
        else:
            selected = c_idx if keep_all_if_less else np.random.choice(c_idx, n_per_class, replace=True)
        idx_selected.extend(selected)
    
    idx_selected = np.array(idx_selected)
    np.random.shuffle(idx_selected)
    return X[idx_selected], y[idx_selected], idx_selected

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Task1_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task1'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Max_Task2_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_task2'][sel])) + _size_penalty(len(sel), len(x))

class HierarchicalFeatureSelection(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Task1_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Max_Task2_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 2. 定制算子 (MOFS-REPLS & DREA-FS) 适配互信息
# ==============================================================
class REPLSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        
        weights = problem.opts['imp_task1']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0] 
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                
                p_nearest_indices = np.argsort(dists)[:self.P]
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0:
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66:
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False

            infills.set("X", X_inf)
        return infills

class DREAFSSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_task1 = problem.opts['imp_task1']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_task1[imp_task1 > 0]) if np.any(imp_task1 > 0) else 0
        important_features = np.where(imp_task1 >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask 
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
                    
        return X_init

class DREA_FS(NSGA2):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 3. Core Evaluation Routine (仅运行两个定制 MOEAs)
# ==============================================================
def calc_mean_std(x):
    return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"

def run_ip_unified_benchmark(features, task1_labels, task2_labels, n_runs=10):
    all_results = []
    
    print("="*70)
    print(f"Starting CUSTOM MOEAs BENCHMARK (MI Aligned): {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Run {run_idx}/{n_runs}...")
        
        le_t1 = LabelEncoder()
        task1_labels_encoded = le_t1.fit_transform(task1_labels)
        
        xtrain, xtest, ytrain, ytest, y2_tr, y2_te = train_test_split(
            features, task1_labels_encoded, task2_labels, 
            test_size=0.3, stratify=task1_labels_encoded, random_state=current_seed
        )
        
        print("  [Pre-compute] Calculating Mutual Information...")
        # 核心修正：使用互信息代替随机森林，保持对齐
        imp_task1 = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
        imp_task2 = mutual_info_classif(xtrain, y2_tr, random_state=current_seed)
        
        opts = {'xt_in': xtrain, 'imp_task1': imp_task1, 'imp_task2': imp_task2}
        problem = HierarchicalFeatureSelection(opts)
        
        ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
        pop_size = len(ref_dirs) # 91
        
        crossover = SinglePointCrossover(prob=0.8)
        mutation = BitflipMutation(prob=0.01)
        
        # 仅加载 2 种定制算法
        algorithms = {
            "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
            "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
        }

        def get_top_k(scores, k):
            idx = np.argsort(scores)[::-1][:k]
            return idx if len(idx) > 0 else np.array([0])

        clf = LinearDiscriminantAnalysis()

        print("  [Optimizing] Running Custom MOEAs...")
        for algo_name, algo in algorithms.items():
            
            start_time = time.time()
            res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
            elapsed_time = time.time() - start_time
            
            final_pop = res.pop.get("X").astype(int)
            scores_algo = final_pop.mean(axis=0)

            for k in range(1, 16):
                idx = get_top_k(scores_algo, k)
                if len(idx) == 0: continue
                
                X_tr_sel, X_te_sel = xtrain[:, idx], xtest[:, idx]

                # Task 1 Evaluation (Fine-grained)
                clf.fit(X_tr_sel, ytrain)
                acc_t1 = clf.score(X_te_sel, ytest)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task1_Fine", "Value": acc_t1,
                    "Time_Seconds": elapsed_time
                })

                # Task 2 Evaluation (Crop vs Non-Crop)
                clf.fit(X_tr_sel, y2_tr)
                acc_t2 = clf.score(X_te_sel, y2_te)
                all_results.append({
                    "Run": run_idx, "Method": algo_name, "K": k,
                    "Task": "Predict_Task2_Crop", "Value": acc_t2,
                    "Time_Seconds": elapsed_time
                })

    # ==============================================================
    # 4. Data Summary & Output
    # ==============================================================
    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("                 CUSTOM MOEAs TABLES [Mean ± Std]")
    print("="*80)
    
    # Task 1 准确率表格
    df_t1 = df_results[df_results["Task"] == "Predict_Task1_Fine"]
    table_t1 = df_t1.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t1 = table_t1[[m for m in METHOD_ORDER if m in table_t1.columns]]
    print("\n[TABLE 1] Task 1: Fine-grained (16-Class) Accuracy")
    print(table_t1)
    table_t1.to_csv("Custom_MOEAs_IP_Task1_LDA.csv")
    
    # Task 2 准确率表格
    df_t2 = df_results[df_results["Task"] == "Predict_Task2_Crop"]
    table_t2 = df_t2.pivot_table(index="K", columns="Method", values="Value", aggfunc=calc_mean_std)
    table_t2 = table_t2[[m for m in METHOD_ORDER if m in table_t2.columns]]
    print("\n[TABLE 2] Task 2: Coarse (Crop/Non-Crop) Accuracy")
    print(table_t2)
    table_t2.to_csv("Custom_MOEAs_IP_Task2_LDA.csv")
    
    # 运行时间表格
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Custom_MOEAs_IP_Execution_Time.csv")
    
    print("\n[+] 印度松林数据集定制算法 MOFS-REPLS 与 DREA-FS 运行完毕！表格已生成。")

# ==============================================================
# 5. Main Execution
# ==============================================================
if __name__ == "__main__":
    DATA_CUBE_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_corrected.mat"
    GT_PATH = "C:/Users/zhan0035/Downloads/datasets/ip/Indian_pines_gt.mat"
    SUBSAMPLE_PER_CLASS = 20   
    KEEP_ALL_IF_LESS = True

    try:
        if not os.path.exists(DATA_CUBE_PATH) or not os.path.exists(GT_PATH):
            raise FileNotFoundError("Indian Pines .mat files not found. Please check paths.")
            
        print("Loading Indian Pines (.mat)...")
        img = _load_mat_first_var(DATA_CUBE_PATH)
        lab = _load_mat_first_var(GT_PATH)

        if lab.ndim == 3 and lab.shape[-1] == 1: lab = lab[:, :, 0]
        lab = lab.astype(int)

        mask = lab > 0
        X = img[mask].astype(float)
        y_fine = lab[mask].astype(int)
        y_crop = make_crop_label_from_indian_pines(y_fine)

        if SUBSAMPLE_PER_CLASS is not None:
            X, y_fine, idx = stratified_subsample_per_class(
                X, y_fine, n_per_class=SUBSAMPLE_PER_CLASS, seed=42, keep_all_if_less=KEEP_ALL_IF_LESS
            )
            y_crop = y_crop[idx]

        X = normalize(X + 1e-8, norm="l1", axis=1)

        run_ip_unified_benchmark(X, y_fine, y_crop, n_runs=30) 

    except Exception as e:
        print(f"\nError encountered: {e}")
        print("Running with Safe Mock Data for architecture verification...")
        set_seed(42)
        X_mock = np.random.rand(500, 200)
        y_fine_mock = np.random.randint(1, 17, 500)
        y_crop_mock = make_crop_label_from_indian_pines(y_fine_mock)
        X_mock = normalize(X_mock + 1e-8, norm="l1", axis=1)
        
        run_ip_unified_benchmark(X_mock, y_fine_mock, y_crop_mock, n_runs=2)

Loading Indian Pines (.mat)...
Starting CUSTOM MOEAs BENCHMARK (MI Aligned): 30 Runs

>>> Run 1/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 2/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 3/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 4/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 5/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 6/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 7/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 8/30...
  [Pre-compute] Calculating Mutual Information...
  [Optimizing] Running Custom MOEAs...

>>> Run 9/30...
  [Pre-compute] Calculating Mutual Information...
